# Bagging intuition with real spam-filter data

Two emails can have the same punctuation and different labels. A filter learns useful patterns from labeled examples. The target is whether the dataset contributors labeled the message **Spam** or **Legitimate**.

This notebook uses the web lesson's same 12 training emails and one held-out email from UCI Spambase. We use two features, the percentages of all characters that are dollar signs and exclamation marks. The full dataset has 4,601 rows and 57 features. The original message text is unavailable.

**Draw three training samples → fit separate trees → predict the same email → combine their answers.**

**Before teaching.** Run all the cells. Reopen the short training cells and the expandable visuals so you can follow each draw and fitted split. The rows and three saved bootstrap samples are embedded here.

**Data:** Spambase by Mark Hopkins, Erik Reeber, George Forman, and Jaap Suermondt (1999). [UCI Machine Learning Repository, DOI 10.24432/C53G6X](https://doi.org/10.24432/C53G6X) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) · [Original field definitions](https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.names)

## From one email to a forest

Draw three training samples, fit one tree per sample with both features available, and predict the same held-out email. Compare the vote with the average of the leaf probabilities. Then see what changes when the forest also samples candidate columns at each split.

An **ensemble** combines models for one task. **Bagging** gives each base model its own bootstrap sample and combines their predictions. In **boosting**, models are added in sequence to improve the ensemble. In **stacking**, a separate model learns to combine other models' predictions. A **cascade** sends more difficult cases to a later model for a closer look.

Next, the implementation notebook uses the full benchmark for tuning, out-of-bag checks, and feature importance.

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text

print(f'NumPy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}')

In [ ]:
# @title Load notebook visuals { display-mode: "form" }
from html import escape
from IPython.display import HTML, display

visual_style = '''
<style>
  .email-visual { color:#253747; background:#f4f8f7; border:1px solid #c9dad7; border-radius:16px;
    padding:clamp(12px, 3vw, 24px); font:16px/1.5 system-ui,sans-serif; box-sizing:border-box; }
  .email-visual h3 { margin:0 0 6px; font-size:clamp(20px, 4vw, 29px); color:#163c3b; }
  .email-visual p { margin:6px 0 13px; }
  .email-visual figure { margin:0; }
  .email-visual details { margin:8px 0 0; border:1px solid #c9dad7; background:#fff; border-radius:10px; }
  .email-visual summary { cursor:pointer; padding:10px 14px; font-weight:700; color:#163c3b; }
  .email-visual summary:focus-visible { outline:2px solid #117069; outline-offset:2px; border-radius:8px; }
  .email-visual details[open] > summary { background:#e5f1ee; border-radius:9px 9px 0 0; }
  .email-visual .visual-body { padding:12px 14px; }
  .email-visual .visual-tickets { display:flex; flex-wrap:wrap; gap:6px; margin:9px 0; }
  .email-visual .visual-ticket { min-width:35px; padding:5px 7px; border:1px solid #b5c5c6;
    border-radius:6px; color:#183e3b; background:#e8f2ee; text-align:center; font-weight:700; }
  .email-visual .visual-repeat { background:#fff0c4; border-color:#96691c; color:#603d04; }
  .email-visual .visual-omitted { background:#f5f7f8; border-style:dashed; color:#4a5964; }
  .email-visual figcaption { color:#364e5e; margin:6px 0; }
  .email-visual .visual-tree-shell { overflow:auto; }
  .email-visual .visual-tree { display:block; width:100%; max-width:820px; min-width:570px; margin:auto; }
  .email-visual .visual-cards { display:grid; grid-template-columns:repeat(auto-fit,minmax(min(100%,225px),1fr)); gap:10px; }
  .email-visual .visual-card { padding:12px; border:1px solid #c9dad7; border-radius:10px; background:#fff; }
  .email-visual .visual-card strong { color:#155854; font-size:17px; }
  .email-visual .visual-grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(54px,1fr)); gap:5px; }
  .email-visual .visual-tree-chip { border:1px solid #b5c5c6; border-radius:7px; text-align:center;
    padding:4px; font-size:12px; background:#f7f9f9; color:#42515b; }
  .email-visual .visual-tree-chip.in-mean { background:#d9f0eb; color:#133e3c; border-color:#62968d; }
  .email-visual .visual-tree-chip b { display:block; font-size:14px; }
</style>
'''

def show_visual(key, title, caption, body):
    display(HTML(f'{visual_style}<section class="email-visual" data-visual="{key}">'
                 f'<h3>{title}</h3><p>{caption}</p>{body}</section>'))

def visual_option(group, state, title, body, opened=False):
    is_open = ' open' if opened else ''
    return (f'<details name="{group}" data-state="{state}"{is_open}>'
            f'<summary>{title}</summary><div class="visual-body">{body}</div></details>')

def visual_tickets(ids, *, duplicates=False, omitted=False):
    seen = set()
    tickets = []
    for row_id in ids:
        repeat = duplicates and row_id in seen
        variant = ' visual-omitted' if omitted else ' visual-repeat' if repeat else ''
        note = ' (repeat)' if repeat else ''
        tickets.append(f'<span class="visual-ticket{variant}">{escape(row_id)}{note}</span>')
        seen.add(row_id)
    return '<div class="visual-tickets">' + ''.join(tickets) + '</div>'

def tree_visual(model, point):
    fitted = model.tree_
    path_nodes = set(model.decision_path(point).indices)
    feature_labels = ['$ %', '! %']
    svg_edges, svg_nodes = [], []

    def visit(node, x, depth, half_span):
        y = 30 + depth * 102
        left, right = fitted.children_left[node], fitted.children_right[node]
        on_path = node in path_nodes
        if left == right:
            label = names[int(model.classes_[fitted.value[node, 0].argmax()])]
        else:
            label = f'{feature_labels[fitted.feature[node]]} ≤ {fitted.threshold[node]:.4f}%'
        count = fitted.weighted_n_node_samples[node]
        node_role = 'draw' if count == 1 else 'draws'
        stroke = '#177669' if on_path else '#9fb1b5'
        fill = '#d9f0eb' if on_path else '#fff'
        svg_nodes.append(
            f'<rect x="{x - 68}" y="{y}" width="136" height="51" rx="8" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="{2.8 if on_path else 1.3}"/>'
            f'<text x="{x}" y="{y + 20}" font-size="15" font-weight="700">{escape(label)}</text>'
            f'<text x="{x}" y="{y + 39}" font-size="13">{count:g} {node_role}</text>'
        )
        if left != right:
            for child, direction, answer in ((left, -1, 'Yes'), (right, 1, 'No')):
                child_x = x + direction * half_span
                stroke = '#177669' if child in path_nodes else '#9fb1b5'
                svg_edges.append(
                    f'<line x1="{x}" y1="{y + 51}" x2="{child_x}" y2="{y + 102}" '
                    f'stroke="{stroke}" stroke-width="{3.8 if child in path_nodes else 1.6}"/>'
                    f'<text x="{(x + child_x) / 2}" y="{y + 70}" font-size="13"'
                    f' stroke="#f4f8f7" stroke-width="4" paint-order="stroke">{answer}</text>'
                )
                visit(child, child_x, depth + 1, half_span / 2)

    visit(0, 330, 0, 162)
    height = 93 + fitted.max_depth * 102
    prediction = names[int(model.predict(point)[0])]
    by_class = dict(zip(model.classes_, model.predict_proba(point)[0]))
    p_spam = by_class.get(1, 0.0)
    description = f'The new email reaches the {prediction} leaf. Spam probability {p_spam:.1%}.'
    return (f'<figure><div class="visual-tree-shell"><svg class="visual-tree" '
            f'viewBox="0 0 660 {height}" role="img" aria-label="{escape(description)}" '
            f'text-anchor="middle" fill="#253747" xmlns="http://www.w3.org/2000/svg">'
            f'{"".join(svg_edges + svg_nodes)}</svg></div><figcaption>'
            f'<strong>{prediction}</strong> · {p_spam:.1%} Spam probability. '
            f'The thick green path follows the new email. The node counts are training draws.</figcaption></figure>')

## Read the training table

The table supplies two measured inputs and the recorded label for each training email. If 2 characters out of 1,000 are dollar signs, the feature is **0.2%**. These data are already in percent units, so no conversion is needed. The E numbers use the original UCI file's row numbers, counting from 1.

We set aside **E574** and supply only its two inputs to the models. We'll reveal its recorded label after the predictions.

In [ ]:
training = pd.DataFrame([
    ('E3590', 0, 0, 0),
    ('E3383', 0.024, 0, 0),
    ('E4256', 0, 0.278, 0),
    ('E2321', 0, 0.089, 0),
    ('E3066', 0, 0.244, 0),
    ('E3776', 0.023, 0, 0),
    ('E779', 0.145, 0.393, 1),
    ('E1375', 0.231, 0.925, 1),
    ('E1748', 0, 0.518, 1),
    ('E1188', 0.312, 0.625, 1),
    ('E319', 0.107, 0.268, 1),
    ('E1153', 0.073, 0.25, 1),
], columns=['email_id', 'dollar_pct', 'exclamation_pct', 'label']).set_index('email_id')

features = ['dollar_pct', 'exclamation_pct']
X = training[features]
y = training['label']
names = {0: 'Legitimate', 1: 'Spam'}
email = pd.DataFrame([(0.034, 0.375)], columns=features, index=['E574'])
bootstrap_seed = 11
tree_seed = 0

display(training[features].assign(recorded_label=y.map(names)), email)

## Draw rows with replacement

Imagine each of the 12 training emails has a ticket. Draw one, copy the whole row into the sample, and put the ticket back. A ticket that goes back can be drawn again. After 12 draws, we have one **bootstrap sample**.

The saved sequence below lets us replay the web lesson. The example starts with two copies of the same email to make the repeat easy to see. The `rng.choice` line shows a separate fresh sample. We keep the saved one for the next three-tree comparison so the fitted paths match the page.

**Try the visual.** Open **1 draw**, then **2 draws**, then **All 12 draws**. Look for repeated and omitted rows. Open **Without replacement** to compare what happens when we shuffle all 12 rows into a tray and each can appear only once.

In [ ]:
draw_ids = [
    ['E3776', 'E3776', 'E1748', 'E4256', 'E1188', 'E1153', 'E2321', 'E4256', 'E1153', 'E3066', 'E3776', 'E1375'],
    ['E3590', 'E319', 'E3776', 'E779', 'E3590', 'E3776', 'E4256', 'E779', 'E2321', 'E1748', 'E1748', 'E3590'],
    ['E1748', 'E319', 'E3383', 'E2321', 'E1153', 'E1748', 'E3776', 'E1748', 'E1375', 'E1375', 'E3383', 'E4256'],
]
demo_draws = np.array(draw_ids)
assert set(demo_draws.ravel()) <= set(X.index)

rng = np.random.default_rng(bootstrap_seed)
fresh_draw = rng.choice(X.index, size=len(X), replace=True)
draw = demo_draws[0]
drawn_ids = pd.Index(draw)
counts = drawn_ids.value_counts().reindex(training.index, fill_value=0)

print(f'{len(draw)} saved draws, {drawn_ids.nunique()} distinct rows')
print('Draw order:', ', '.join(drawn_ids))
print('Omitted:', ', '.join(counts.index[counts == 0]) or 'none')
print('One fresh sample:', ', '.join(fresh_draw))

In [ ]:
# @title Explore the training tray { display-mode: "form" }
def draw_panel(ids, complete=False):
    body = '<strong>Draw order</strong>' + visual_tickets(ids.tolist(), duplicates=True)
    draw_count, email_count = len(ids), ids.nunique()
    draw_label = 'draw' if draw_count == 1 else 'draws'
    email_label = 'email' if email_count == 1 else 'emails'
    body += f'<p>{draw_count} {draw_label} · {email_count} distinct {email_label}</p>'
    if complete:
        omitted = training.index.difference(ids, sort=False)
        if len(omitted):
            body += '<strong>Still on the desk, absent from this tray</strong>'
            body += visual_tickets(omitted.tolist(), omitted=True)
        else:
            body += '<p>Every email appears once. The order has changed, and the records are all still there.</p>'
    return body

shuffled_ids = training.index[
    np.random.default_rng(bootstrap_seed).choice(len(X), size=len(X), replace=False)
]
draw_cards = (
    visual_option('bootstrap-stages', 'draw-1', '1 draw', draw_panel(drawn_ids[:1]), opened=True)
    + visual_option('bootstrap-stages', 'draw-2', '2 draws', draw_panel(drawn_ids[:2]))
    + visual_option('bootstrap-stages', 'draw-all', 'All 12 draws', draw_panel(drawn_ids, complete=True))
    + visual_option('bootstrap-stages', 'no-replacement', 'Without replacement',
                    draw_panel(shuffled_ids, complete=True))
)
show_visual('bootstrap-draws', 'Build one training tray',
            'Copy the drawn record into the tray and return the original to the desk. '
            'Open a step to see what is in the tray.', draw_cards)

## Fit one tree and predict E574

The tree learns its questions from the drawn rows and their labels. If a row appears twice, its label contributes twice. This tree can ask up to two questions on each path. Find its first cutoff and use E574's feature value to choose Yes or No.

In [ ]:
tree = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=tree_seed)
tree.fit(X.loc[draw], y.loc[draw])
prediction = int(tree.predict(email)[0])

print(f'Tree 1 predicts {names[prediction]} for the new email.')
print(export_text(tree, feature_names=features, decimals=4))

In [ ]:
# @title Trace one decision tree { display-mode: "form" }
show_visual('one-tree', 'One bootstrap sample, one tree, one answer',
            'Follow the green path for E574. The cutoff and the email input use the same percent units.',
            tree_visual(tree, email))

## Bootstrap, fit, predict, then combine

Each tree fits a different saved sample from the same 12 training rows. The input for their prediction is always E574. Open two trees and check whether they learned the same questions.

A tree estimates the probability of Spam using the fraction of spam draws in the leaf the email reaches. The code places each probability in its original 0 or 1 class column. If one sample contains only one class, the other column stays zero.

In [ ]:
trees = []
probabilities = np.zeros((len(demo_draws), len(names)))
demo_results = []

for index, sample in enumerate(demo_draws):
    model = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=tree_seed)
    model.fit(X.loc[sample], y.loc[sample])
    trees.append(model)
    probabilities[index, model.classes_] = model.predict_proba(email)[0]
    demo_results.append({
        'tree': index + 1,
        'distinct rows': len(np.unique(sample)),
        'root feature': features[model.tree_.feature[0]],
        'root cut': model.tree_.threshold[0],
        'class': names[int(model.predict(email)[0])],
        'P(Spam)': probabilities[index, 1],
    })

display(pd.DataFrame(demo_results).round(4))

In [ ]:
# @title Compare the three fitted trees { display-mode: "form" }
tree_cards = []
for index, (sample, model) in enumerate(zip(demo_draws, trees), start=1):
    ids = pd.Index(sample)
    tree_cards.append(visual_option(
        'email-trees', f'tree-{index}', f'Tree {index}',
        f'<strong>Training sample · {ids.nunique()} distinct emails</strong>'
        + visual_tickets(ids.tolist(), duplicates=True) + tree_visual(model, email),
        opened=index == 1,
    ))
show_visual('manual-forest', 'Three trees predict the same email',
            'Switch trees to see the sample it trains on and the path for E574.',
            ''.join(tree_cards))

## Combine the trees' predictions

**Hard vote.** Each tree contributes one prediction. One vote for each class is a tie. When there are three predictions, pick the class with the most votes. The count of tree votes describes agreement among the trees.

**Average probabilities.** Each tree contributes the Spam and Legitimate fractions from the leaf it reached. Average each class's probabilities across trees, then choose the highest mean. The examples below show both rules on the same E574. Exact ties in the code's selected class resolve to the smaller encoded class.

**Read the observed outcome.** The last detail reveals the Spam or Legitimate label recorded in the dataset. This one email helps us understand aggregation. Use many unseen emails to evaluate accuracy.

In [ ]:
hard_predictions = np.array([model.predict(email)[0] for model in trees])
votes = np.bincount(hard_predictions, minlength=len(names))
mean_probabilities = probabilities.mean(axis=0)

print(f'Hard vote: Legitimate {votes[0]}, Spam {votes[1]}. Result: {names[int(votes.argmax())]}')
print(f'Mean P(Spam): {mean_probabilities[1]:.4f}. Result: {names[int(mean_probabilities.argmax())]}')
np.testing.assert_array_equal(hard_predictions, [0, 1, 1])
np.testing.assert_allclose(probabilities[:, 1], [0, 1, 0.6666666666666666], atol=1e-9)

In [ ]:
# @title Compare the two ways to combine answers { display-mode: "form" }
vote_text = ' + '.join(names[int(prediction)] for prediction in hard_predictions)
leaf_fractions = []
for model, sample in zip(trees, demo_draws):
    leaf = model.apply(email)[0]
    in_leaf = model.apply(X.loc[sample]) == leaf
    rows = y.loc[sample].to_numpy()[in_leaf]
    leaf_fractions.append(f'{np.count_nonzero(rows == 1)}/{len(rows)}')
formula = ' + '.join(leaf_fractions)

show_visual('aggregation', 'Turn three predictions into one',
            'All three trees predict the same held-out email. Read across to compare giving each tree one vote '
            'with using the actual mix of training records in its leaf.',
            f'<div class="visual-cards"><div class="visual-card"><strong>Hard vote</strong>'
            f'<p>{vote_text}</p><p>{votes[0]} Legitimate · {votes[1]} Spam → '
            f'<b>{names[int(votes.argmax())]}</b></p></div>'
            f'<div class="visual-card"><strong>Average Spam probability</strong>'
            f'<p>({formula}) ÷ {len(trees)} = {mean_probabilities[1]:.1%}</p>'
            f'<p>Higher mean probability → <b>{names[int(mean_probabilities.argmax())]}</b></p></div></div>'
            + visual_option('held-out-label', 'recorded-label', 'Reveal the recorded label',
                            '<p>UCI labels E574 <strong>Spam</strong>.</p>'))

## Predict a number with the same idea

This separate math example uses electricity demand. Suppose three regression trees predict 9, 11, and 13 kWh for the same time period. The target is a number, so a random forest regressor averages the predictions directly. This example uses illustrative values and is separate from the Spambase data.

In [ ]:
# @title Combine three electricity demand predictions { display-mode: "form" }
demand_predictions = np.array([9, 11, 13])
cards = ''.join(
    f'<div class="visual-card"><strong>Tree {i}</strong><p>{kwh} kWh</p></div>'
    for i, kwh in enumerate(demand_predictions, 1)
)
show_visual(
    'demand-regression', 'How much electricity?',
    'Open the answer to average the three illustrative predictions.',
    f'<div class="visual-cards">{cards}</div>'
    + visual_option('demand-regression', 'mean', 'Reveal the mean',
                    f'<p>(9 + 11 + 13) ÷ 3 = <strong>{demand_predictions.mean():g} kWh</strong>.</p>'),
)

## See what a random forest changes

The previous example let every split consider both inputs. A random forest draws a fresh subset of candidate features at each split, as well as a bootstrap sample for each tree. That gives a strong feature less opportunity to drive the same splits in every tree. Here `max_features=1` sets the candidate budget to one feature. The splitter can inspect additional candidates if it needs them to find a valid split.

This 64-tree implementation draws its own samples. Open one, eight, and all 64 trees to compare their running average. Each tile shows a fitted tree's Spam probability for the same held-out message.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=64, max_depth=2, max_features=1,
    bootstrap=True, random_state=bootstrap_seed, n_jobs=1,
).fit(X, y)

# This example encodes the two classes as 0 and 1, in the same column order.
email_array = email.to_numpy()
tree_probabilities = np.array([estimator.predict_proba(email_array)[0]
                               for estimator in forest.estimators_])
forest_probabilities = forest.predict_proba(email)[0]
np.testing.assert_allclose(tree_probabilities.mean(axis=0), forest_probabilities)
display(pd.Series(forest_probabilities, index=[names[int(c)] for c in forest.classes_],
                  name='Mean forest probability').round(4).to_frame().T)
print('Random Forest prediction:', names[int(forest.predict(email)[0])])

In [ ]:
# @title Explore the fitted Random Forest { display-mode: "form" }
def forest_panel(limit):
    chips = []
    for i, p_spam in enumerate(tree_probabilities[:limit, 1], start=1):
        chips.append(
            f'<span class="visual-tree-chip in-mean" aria-label="Tree {i}, {p_spam:.1%} Spam">'
            f'Tree {i}<b>{p_spam:.0%}</b></span>'
        )
    mean = tree_probabilities[:limit].mean(axis=0)
    route = names[int(forest.classes_[mean.argmax()])]
    body = (f'<p><strong>{limit} {"tree" if limit == 1 else "trees"} included</strong> · '
            f'Average Spam probability {mean[1]:.1%} → <strong>{route}</strong></p>'
            f'<div class="visual-grid">{"".join(chips)}</div>')
    if limit == 1:
        body += tree_visual(forest.estimators_[0], email_array)
    return body

forest_cards = ''.join(
    visual_option('email-forest-sizes', f'forest-{limit}', title, forest_panel(limit), opened=limit == 1)
    for limit, title in ((1, 'First tree'), (8, 'First 8 trees'), (64, 'All 64 trees'))
)
show_visual('random-forest', 'Inside the fitted Random Forest',
            'Each tile shows one fitted tree’s Spam probability for our new email. '
            'Open the next step to include more trees in the average.',
            forest_cards)

## What happens when training changes?

Deep trees can learn subtle patterns and also quirks specific to their training sample. A shallow tree may miss a useful combination of features. Use unseen examples to check whether extra splits make predictions better away from the training data.

Override one real training label in a copy of the data and refit the same three bootstrap samples. A tree that never drew the edited row sees exactly the same training input. Another tree can change. The source records and the test email remain fixed.

In [ ]:
# @title Change one historical email label { display-mode: "form" }
def compare_label_change(ticket_id):
    appearances = (demo_draws == ticket_id).sum(axis=1)
    revised_labels = y.copy()
    revised_labels.loc[ticket_id] = 1 - int(y.loc[ticket_id])
    revised_spam = []
    for sample in demo_draws:
        revised_tree = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=tree_seed)
        revised_tree.fit(X.loc[sample], revised_labels.loc[sample])
        by_class = dict(zip(revised_tree.classes_, revised_tree.predict_proba(email)[0]))
        revised_spam.append(by_class.get(1, 0.0))

    original_spam = probabilities[:, 1]
    cards = []
    for i, (copies, before, after) in enumerate(zip(appearances, original_spam, revised_spam), 1):
        copy_label = 'copy' if copies == 1 else 'copies'
        cards.append(
            f'<div class="visual-card"><strong>Tree {i}</strong><p>{copies} {copy_label} of {ticket_id}</p>'
            f'<p>Before&nbsp;{before:.1%}<br>After&nbsp;{after:.1%} Spam</p></div>'
        )
    totals = (
        f'<p>Mean Spam probability across the three trees: '
        f'<strong>{original_spam.mean():.1%} → {np.mean(revised_spam):.1%}</strong>.</p>'
    )
    return f'<div class="visual-cards">{"".join(cards)}</div>{totals}'

label_change_ids = ['E3383', 'E3590']
label_change_cases = ''.join(
    visual_option('label-change', ticket_id, f'Change {ticket_id}’s recorded label',
                  compare_label_change(ticket_id))
    for ticket_id in label_change_ids
)
show_visual(
    'label-change', 'Which trees can notice the label change?',
    'In each example, only that training email label is flipped for the demonstration. '
    'A tree whose tray has zero copies still sees the same training data. '
    'A tree whose tray includes the ticket can change its answer, and sometimes gives the same answer anyway.',
    label_change_cases,
)

## Why averaging can reduce variance

Hold the same target email fixed. Across repeated training sets, two trees might change their Spam scores in opposite directions. Those changes partly cancel in an average. If their errors tend to move in the same direction, less variation cancels.

To separate the sources of squared error, use an illustrative binary outcome. Suppose the true long-run Spam rate for emails with a fixed feature pattern is 0.50, and two equally likely training sets give scores of 0.40 and 0.80. The expected squared error is the sum of squared bias, variance, and irreducible noise. These values illustrate the formula. They are not an estimate from the Spambase benchmark.

In [ ]:
# @title Work through the squared error { display-mode: "form" }
predictions = np.array([0.4, 0.8])
outcomes = np.array([0, 1])
fitted_mean, true_rate = predictions.mean(), outcomes.mean()
bias_squared = (fitted_mean - true_rate) ** 2
variance = predictions.var()
noise = outcomes.var()
expected_error = np.mean((predictions[:, None] - outcomes) ** 2)
np.testing.assert_allclose(expected_error, bias_squared + variance + noise)

components = (
    ('Squared bias', '(0.60 − 0.50)² = 0.01. The mean score differs from the true rate.'),
    ('Prediction variance', '[(0.40 − 0.60)² + (0.80 − 0.60)²] ÷ 2 = 0.04. The fitted score changes.'),
    ('Irreducible noise', '[(0 − 0.50)² + (1 − 0.50)²] ÷ 2 = 0.25. The true label can be either class.'),
)
cards = ''.join(f'<div class="visual-card"><strong>{name}</strong><p>{explanation}</p></div>'
                for name, explanation in components)
show_visual('squared-error', 'What makes a probability estimate miss?',
            'Reveal the three terms of the expected squared error for this illustrative case.',
            visual_option('squared-error', 'components', 'Reveal bias, variance, and noise',
                          cards + f'<p><strong>0.01 + 0.04 + 0.25 = {expected_error:.2f}</strong></p>'))